In [1]:
import re
import html
import warnings
from pathlib import Path

import holidays
import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.corpus import wordnet
from nltk import pos_tag
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from langdetect import detect, DetectorFactory, LangDetectException

DetectorFactory.seed = 42
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)

# Download resource NLTK yang dibutuhkan (butuh koneksi internet sekali saja)
for pkg in ["stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger_eng"]:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print(f"Gagal mengunduh '{pkg}': {e}")


In [2]:
news_raw = pd.read_csv('../data/raw/cnbc_with_published.csv')
bi_raw = pd.read_csv('../data/raw/bi-usd-rate.csv')

print("Berita geopolitik :", news_raw.shape)
print("Kurs USD BI       :", bi_raw.shape)
news_raw.head(5)


Berita geopolitik : (16733, 13)
Kurs USD BI       : (1202, 5)


,keyword_matched,title,url,preview_summary,full_text,date_raw,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status
0,geopolitical risk,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,https://www.cnbc.com/2026/09/08/global-shipping-iran-war-hormuz-rules.html?&qsearchterm=geopolitical risk,Global maritime authorities have warned that the emergence of “parallel systems” threatens to create a two-tier stru...,"Global maritime authorities have warned that the emergence of ""parallel systems"" threatens to create a two-tier stru...",9/8/2026 2:28:41 PM,Markets,2026-09-08T07:28:41+0000,2026-09-08,07:28 AM,0.0,2026-09-08T07:28:41+0000,ok
1,geopolitical risk,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,https://www.cnbc.com/video/2026/07/31/morning-call-sheet-ai-rebound-meets-rising-geopolitical-risks.html?&qsearchter...,"Peter Tchir, Head of Macro Strategy at Academy Securities, Thomas Martin, Senior Portfolio Manager at GLOBALT Invest...",NaN,7/31/2026 5:58:53 PM,Morning Call,NaN,NaN,NaN,NaN,NaN,not_found
2,geopolitical risk,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",https://www.cnbc.com/2026/09/08/global-markets-shrug-off-shocks-hsbc-sees-what-could-break-the-streak.html?&qsearcht...,"Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developments that could ...","In this article Global markets have shrugged off a barrage of shocks in recent years, but HSBC sees several developm...",9/8/2026 11:02:43 AM,World Markets,2026-09-08T04:02:43+0000,2026-09-08,04:02 AM,0.0,2026-09-08T04:02:43+0000,ok
3,geopolitical risk,"Yen hovers near seven-month high, dollar steadies",https://www.cnbc.com/2026/09/08/yen-extends-rally-to-new-seven-month-high-dollar-subdued-ahead-of-cpi.html?&qsearcht...,"The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed against major peer...","In this article The Japanese yen hovered near ​a seven-month high on Tuesday, while the dollar was little changed ag...",9/8/2026 11:17:07 AM,Currencies,2026-09-08T04:17:07+0000,2026-09-08,04:17 AM,0.0,2026-09-08T04:17:07+0000,ok
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",6/17/2026 2:28:36 PM,Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok


In [3]:
news_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 16733 entries, 0 to 16732
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   keyword_matched     16733 non-null  str    
 1   title               16733 non-null  str    
 2   url                 16733 non-null  str    
 3   preview_summary     16733 non-null  str    
 4   full_text           13431 non-null  str    
 5   date_raw            16733 non-null  str    
 6   section             16649 non-null  str    
 7   published_raw       15239 non-null  str    
 8   published_date      15239 non-null  str    
 9   published_time      15239 non-null  str    
 10  published_timezone  15239 non-null  float64
 11  published_iso       15239 non-null  str    
 12  scrape_status       16733 non-null  str    
dtypes: float64(1), str(12)
memory usage: 1.7 MB


In [4]:
# Parse timestamp mentah GEO dan pisahkan tanggal & jam dengan timezone WIB.
news_raw["date_raw_dt"] = pd.to_datetime(news_raw["date_raw"], errors="coerce")
news_raw["date_raw_dt_wib"] = (
    pd.to_datetime(news_raw["date_raw_dt"], errors="coerce", utc=True)
    .dt.tz_convert("Asia/Jakarta")
    .dt.tz_localize(None)
)
news_raw["tanggal"] = news_raw["date_raw_dt_wib"].dt.strftime("%m/%d/%Y")
news_raw["jam"] = news_raw["date_raw_dt_wib"].dt.strftime("%I:%M %p")
news_raw = news_raw.drop(columns=["date_raw"], errors="ignore")

print("Kolom Berita Geopolitik:")
news_raw[["title", "tanggal", "jam", "date_raw_dt_wib"]].head(5)


Kolom Berita Geopolitik:


,title,tanggal,jam,date_raw_dt_wib
0,Global shipping authorities warn of maritime trade breakdown amidgeopoliticalturmoil,09/08/2026,09:28 PM,2026-09-08 21:28:41
1,Morning Call Sheet: AI rebound meets risinggeopoliticalrisks,08/01/2026,12:58 AM,2026-08-01 00:58:53
2,"Global markets keep shrugging off shocks. Here’s what could break that streak, according to HSBC",09/08/2026,06:02 PM,2026-09-08 18:02:43
3,"Yen hovers near seven-month high, dollar steadies",09/08/2026,06:17 PM,2026-09-08 18:17:07
4,Central banks are bringing gold reserves home asgeopoliticalrisks rise,06/17/2026,09:28 PM,2026-06-17 21:28:36


In [5]:
bi_raw["Tanggal_dt"] = pd.to_datetime(bi_raw["Tanggal"], errors="coerce")
bi_raw["tanggal"] = bi_raw["Tanggal_dt"].dt.strftime("%m/%d/%Y")
bi_raw["jam"] = bi_raw["Tanggal_dt"].dt.strftime("%I:%M %p")
bi_raw = bi_raw.drop(
    columns=["Tanggal"],
    errors="ignore"
)

print("Kolom Kurs BI:")
bi_raw.head(5)

Kolom Kurs BI:


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM


# Data Cleaning

## 1. Filter Dataset


In [6]:
#delete duplicate rows
def word_count(text):
    if not isinstance(text, str):
        return 0
    return len(text.split())

news = news_raw.copy()
# Ukuran data awal
print(f"Baris awal                         : {len(news_raw)}")

# Filter rentang tanggal inklusif: 1 September 2021 - 1 September 2026
start_date = pd.Timestamp("2021-09-01")
end_date = pd.Timestamp("2026-09-01 23:59:59")
news = news[news["date_raw_dt"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang 1 September 2021 - 1 September 2026 : {len(news)}")

# Setelah di drop URL
news = news.drop_duplicates(subset=["url"], keep="first")
print(f"Setelah dedup url                  : {len(news)}")

# Setelah di drop title + date_raw
news = news.drop_duplicates(subset=["title", "tanggal", "jam"], keep="first")
print(f"Setelah dedup title+date_raw       : {len(news)}")

Baris awal                         : 16733
Setelah filter rentang 1 September 2021 - 1 September 2026 : 8871
Setelah dedup url                  : 8871
Setelah dedup title+date_raw       : 7084


In [7]:
# Filter full_text yang kosong

# Buang baris yang tidak memiliki isi teks pada ketiga sumber konten.
text_columns = ["title", "preview_summary", "full_text"]
text_content = news[text_columns].fillna("").astype(str).agg(" ".join, axis=1).str.strip()
mask_empty = text_content.eq("")
news = news.loc[~mask_empty].copy()
print(f"Setelah drop teks kosong             : {len(news)}")

# Buang item multimedia berdasarkan URL, judul, dan section.
non_article_pattern = r"\b(video|videos|image|images|photo|photos|foto|gambar|audio|podcast|gallery|galeri|live)\b|\.(mp4|mov|avi|mp3|wav|jpg|jpeg|png)(?:$|[?#])"
content_metadata = news[["url", "title", "section"]].fillna("").astype(str).agg(" ".join, axis=1)
mask_non_article = content_metadata.str.contains(non_article_pattern, case=False, regex=True, na=False)
news = news.loc[~mask_non_article].copy()
print(f"Setelah drop video/gambar/audio      : {len(news)}")

news.head(5)

Setelah drop teks kosong             : 7084
Setelah drop video/gambar/audio      : 6214


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
4,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
8,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
9,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM
13,geopolitical risk,"Citi Wealth warns markets may be ‘uncomfortably strong’ amid mountinggeopolitical, inflation risks",https://www.cnbc.com/2026/05/18/citi-wealth-warns-markets-may-be-uncomfortably-strong-as-risks-mount.html?&qsearchte...,"Global markets may be due for a period of consolidation after a sharp rally in equities, even as the longer-term out...",NaN,Pro: Analysis,2026-05-18T05:37:18+0000,2026-05-18,05:37 AM,0.0,2026-05-18T05:37:18+0000,ok,2026-05-18 12:37:18,2026-05-18 19:37:18,05/18/2026,07:37 PM
15,geopolitical risk,Where fixed income investors are finding yield asgeopoliticalriskrattles markets,https://www.cnbc.com/2026/04/10/where-fixed-income-investors-are-finding-yield-as-geopolitical-risk-rattles-markets....,"As the Iran war shakes up markets, strategists say there are still plenty of sources of relatively safe yield for in...",NaN,Pro: Income Investing,2026-04-10T19:25:17+0000,2026-04-10,07:25 PM,0.0,2026-04-10T19:25:17+0000,ok,2026-04-11 02:25:17,2026-04-11 09:25:17,04/11/2026,09:25 AM


In [8]:
# Panjang text minimal 350

effective_text = news["full_text"].fillna(news["preview_summary"])
news = news.loc[effective_text.apply(word_count) >= 350]
print(f"Setelah filter panjang teks min     : {len(news)}")

Setelah filter panjang teks min     : 4349


In [9]:
#Filter keyword yang relevan dengan geopolitik dan ekonomi
RELEVANCE_KEYWORDS = [
    "geopolitics",
    "geopolitical risk",
    "geopolitical tensions",
    "geopolitical fragmentation",
    "armed conflict",
    "global conflict",
    "international conflict",
    "military tensions",
    "national security",
    "sanctions",
    "trade war",
    "tariffs",
    "embargo",
    "export controls",
    "protectionism",
    "supply chain disruption",
    "diplomacy",
    "foreign policy",
    "nato",
    "brics",
    "opec",
    "g7",
]

EXCLUDE_SECTIONS = [
    "sports", "entertainment", "television", "lifestyle", "travel",
    "food retail", "beer, wine & spirits", "college", "modern medicine",
    "life changes", "fashion", "celebrity",
]

def is_relevant(row):
    """Berita dianggap relevan jika keyword_matched terisi ATAU teks memuat
    salah satu kata kunci geopolitik/ekonomi pada RELEVANCE_KEYWORDS."""
    if isinstance(row["keyword_matched"], str) and row["keyword_matched"].strip():
        return True
    haystack = f"{row.get('title', '')} {row.get('preview_summary', '')}".lower()
    return any(kw in haystack for kw in RELEVANCE_KEYWORDS)

mask_relevant = news.apply(is_relevant, axis=1)
news = news.loc[mask_relevant]

print(f"Setelah filter relevansi kata kunci : {len(news)}")

# 4.5 Filter kategori/section yang tidak relevan
section_lower = news["section"].fillna("").str.lower()
mask_section = ~section_lower.isin(EXCLUDE_SECTIONS)
news = news.loc[mask_section].reset_index(drop=True)

print(f"Setelah filter kategori/section     : {len(news)}")

news.head(3)


Setelah filter relevansi kata kunci : 4349
Setelah filter kategori/section     : 4336


,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,published_iso,scrape_status,date_raw_dt,date_raw_dt_wib,tanggal,jam
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17T07:28:36+0000,2026-06-17,07:28 AM,0.0,2026-06-17T07:28:36+0000,ok,2026-06-17 14:28:36,2026-06-17 21:28:36,06/17/2026,09:28 PM
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27T02:49:03+0000,2026-08-27,02:49 AM,0.0,2026-08-27T02:49:03+0000,ok,2026-08-27 09:49:03,2026-08-27 16:49:03,08/27/2026,04:49 PM
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-20T23:01:01+0000,2026-07-20,11:01 PM,0.0,2026-07-20T23:01:01+0000,ok,2026-07-21 06:01:01,2026-07-21 13:01:01,07/21/2026,01:01 PM


In [10]:
# delete missing values
jumlah_nan = news["full_text"].isna().sum()

print("Jumlah full_text NaN:", jumlah_nan)

news_nan = news.loc[news["full_text"].isna()]

news_nan[
    ["url", "title", "preview_summary", "full_text"]
].head()

news = news.dropna(subset=["full_text"]).reset_index(drop=True)

print("Jumlah baris setelah drop NaN:", len(news))
print("Sisa NaN pada full_text:", news["full_text"].isna().sum())

Jumlah full_text NaN: 0
Jumlah baris setelah drop NaN: 4336
Sisa NaN pada full_text: 0


## 2. Cleaning Data News

In [11]:
#menggabungkan title, preview, dan full_text

# Tidak ada nilai kosong yang mengganggu penggabungan teks.
news[["title", "preview_summary", "full_text"]] = (
    news[["title", "preview_summary", "full_text"]].fillna("")
)

# Gabungkan title + preview_summary + full_text.
news["combined_raw"] = (
    ((news["title"] + " ") * 2)
    + news["preview_summary"] + " "
    + news["full_text"]
)

print(f"Jumlah baris: {len(news)}")
news[["title", "combined_raw"]].head(3)

Jumlah baris: 4336


,title,combined_raw
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...


In [12]:
import unicodedata
from collections import Counter

symbol_counter = Counter()
for text in news["combined_raw"]:
    for ch in text:
        if not (ch.isalnum() or ch.isspace()):
            symbol_counter[ch] += 1

symbol_df = pd.DataFrame(
    [
        (ch, unicodedata.name(ch, "UNKNOWN"), count)
        for ch, count in symbol_counter.most_common()
    ],
    columns=["simbol", "nama_unicode", "jumlah"],
)

print(f"Jumlah jenis simbol unik pada combined_raw: {len(symbol_df)}")
symbol_df

Jumlah jenis simbol unik pada combined_raw: 54


,simbol,nama_unicode,jumlah
0,.,FULL STOP,232989
1,",",COMMA,203423
2,"""",QUOTATION MARK,88912
3,',APOSTROPHE,60351
4,-,HYPHEN-MINUS,41818
5,%,PERCENT SIGN,18915
6,—,EM DASH,10379
7,$,DOLLAR SIGN,9806
8,’,RIGHT SINGLE QUOTATION MARK,6782
9,:,COLON,5100


In [13]:
#cleaning dari tanda baca yang tidak berguna

URL_RE = re.compile(r"http\S+|www\.\S+")
EMAIL_RE = re.compile(r"\S+@\S+\.\S+")
HTML_TAG_RE = re.compile(r"<[^>]+>")
EMOJI_RE = re.compile(
    "["
    "\U0001F300-\U0001FAFF"
    "\U00002700-\U000027BF"
    "\U0001F1E6-\U0001F1FF"
    "\U00002600-\U000026FF"
    "]+",
    flags=re.UNICODE,
)

# Hanya simbol mata uang yang diubah jadi kata.
# Simbol lain (&, +, tanda baca umum) dibuang di NON_ALNUM_RE.
CURRENCY_WORDS = {
    "%": "percent",
    "$": "dollar",
    "£": "pound",
    "€": "euro",
    "¥": "yen",
}
CURRENCY_RE = re.compile("|".join(re.escape(sym) for sym in CURRENCY_WORDS))
NON_ALNUM_RE = re.compile(r"[^a-zA-Z0-9\s]")  # simbol/tanda baca sisa dibuang; huruf & angka dipertahankan
MULTI_SPACE_RE = re.compile(r"\s+")


def normalize_text(text: str) -> str:
    """Hapus HTML/URL/email/emoji, ubah simbol mata uang jadi kata, angka dipertahankan."""
    if not isinstance(text, str) or not text.strip():
        return ""
    t = html.unescape(text)
    t = HTML_TAG_RE.sub(" ", t)
    t = URL_RE.sub(" ", t)
    t = EMAIL_RE.sub(" ", t)
    t = EMOJI_RE.sub(" ", t)
    t = CURRENCY_RE.sub(lambda m: f" {CURRENCY_WORDS[m.group(0)]} ", t)
    t = NON_ALNUM_RE.sub(" ", t)
    t = MULTI_SPACE_RE.sub(" ", t).strip()
    return t


news["normalized_text"] = news["combined_raw"].apply(normalize_text)

news[["combined_raw", "normalized_text"]].head(3)

,combined_raw,normalized_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...
1,Oil rises 2% as White House says no US-Iran talks happening Oil rises 2% as White House says no US-Iran talks happen...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ... Jamie Dimon says...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...


In [14]:
# lowercasing
news["lower_text"] = news["normalized_text"].str.lower()

news[["normalized_text", "lower_text"]].head(3)

,normalized_text,lower_text
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...
1,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...
2,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...


In [15]:
#memastikan bahasanya english semua
def detect_language(text: str) -> str:
    if not isinstance(text, str) or len(text.strip()) < 20:
        return "unknown"
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


news["detected_lang"] = news["lower_text"].apply(detect_language)

print("Distribusi bahasa terdeteksi:")
print(news["detected_lang"].value_counts().head(10))

baris_sebelum = len(news)
news = news.loc[news["detected_lang"] == "en"].reset_index(drop=True)
print(f"\nSetelah filter bahasa Inggris (en) : {len(news)} (dibuang {baris_sebelum - len(news)} baris)")

news[["lower_text", "detected_lang"]].head(3)

Distribusi bahasa terdeteksi:
detected_lang
en    4336
Name: count, dtype: int64

Setelah filter bahasa Inggris (en) : 4336 (dibuang 0 baris)


,lower_text,detected_lang
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,en
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,en
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,en


## 3. Cleaning Data Kurs

In [16]:
# Gabungkan tanggal dan jam untuk satu kolom datetime kanonik.
bi_raw["Tanggal"] = pd.to_datetime(
    bi_raw["tanggal"].astype(str).str.strip() + " " + bi_raw["jam"].astype(str).str.strip(),
    format="%m/%d/%Y %I:%M %p",
    errors="coerce",
)

print(bi_raw.shape)
bi_raw.head(3)

(1202, 8)


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28


In [17]:
def clean_numeric(series: pd.Series) -> pd.Series:
    """Hapus pemisah ribuan dan konversi nilai ke float."""
    cleaned = series.astype(str).str.replace(",", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")



bi_raw["Kurs Jual"] = clean_numeric(bi_raw["Kurs Jual"])
bi_raw["Kurs Beli"] = clean_numeric(bi_raw["Kurs Beli"])
bi_raw["NO"] = pd.to_numeric(bi_raw["NO"], errors="coerce")
bi_raw["Nilai"] = pd.to_numeric(bi_raw["Nilai"], errors="coerce")

print(f"Baris awal BI                 : {len(bi_raw)}")

# Pertahankan hanya rentang 1 September 2021 sampai 1 September 2026.
bi = bi_raw.loc[bi_raw["Tanggal"].between(start_date, end_date, inclusive="both")]
print(f"Setelah filter rentang tanggal: {len(bi)}")

mask_valid_numeric = (
    bi["Kurs Jual"].notna()
    & bi["Kurs Beli"].notna()
    & (bi["Kurs Jual"] > 0)
    & (bi["Kurs Beli"] > 0)
)
bi = bi.loc[mask_valid_numeric]
print(f"Setelah filter numerik valid  : {len(bi)}")

mask_valid_spread = bi["Kurs Jual"] > bi["Kurs Beli"]
bi = bi.loc[mask_valid_spread]
print(f"Setelah filter Jual > Beli    : {len(bi)}")

bi = (
    bi.sort_values("Tanggal")
    .drop_duplicates(subset=["Tanggal"], keep="last")
    .reset_index(drop=True)
)
print(f"Setelah dedup tanggal         : {len(bi)}")

bi.head(3)

Baris awal BI                 : 1202
Setelah filter rentang tanggal: 1202
Setelah filter numerik valid  : 1202
Setelah filter Jual > Beli    : 1202
Setelah dedup tanggal         : 1202


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal
0,1202,1,14377.53,14234.47,2021-09-01,09/01/2021,12:00 AM,2021-09-01
1,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02
2,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03


# Preprocessing

In [18]:
#stopword removal
CUSTOM_STOPWORDS = {
    "cnbc", "article", "read", "also", "said", "says", "say",
    "would", "could", "also", "one", "two", "get", "got",
}
STOPWORDS_EN = set(stopwords.words("english")) | CUSTOM_STOPWORDS


def remove_stopwords(text: str) -> list:
    # Tokenisasi split-spasi lalu buang stopword Bahasa Inggris & token 1 karakter.
    if not isinstance(text, str) or not text.strip():
        return []
    tokens = text.split()
    return [tok for tok in tokens if tok not in STOPWORDS_EN and len(tok) > 1]


news["tokens_no_stopwords"] = news["lower_text"].apply(remove_stopwords)
news["text_no_stopwords"] = news["tokens_no_stopwords"].apply(" ".join)

avg_before = news["lower_text"].apply(word_count).mean()
avg_after = news["tokens_no_stopwords"].apply(len).mean()
print(f"Rata-rata jumlah kata sebelum stopword removal : {avg_before:.1f}")
print(f"Rata-rata jumlah kata sesudah stopword removal  : {avg_after:.1f}")

news[["lower_text", "text_no_stopwords"]].head(3)

Rata-rata jumlah kata sebelum stopword removal : 862.4
Rata-rata jumlah kata sesudah stopword removal  : 494.5


,lower_text,text_no_stopwords
0,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...
1,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...
2,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...


In [19]:
#lemmatization (POS aware)
lemmatizer = WordNetLemmatizer()

POS_MAP = {
    "J": wordnet.ADJ,
    "V": wordnet.VERB,
    "N": wordnet.NOUN,
    "R": wordnet.ADV,
}


def get_wordnet_pos(treebank_tag: str) -> str:
    # Tag POS Penn Treebank (huruf pertama) ke format POS WordNet; default kata benda.
    return POS_MAP.get(treebank_tag[0], wordnet.NOUN)


def lemmatize_tokens(tokens: list) -> list:
    # Lemmatisasi tiap token berdasarkan POS tag-nya (POS-aware lemmatization).
    if not tokens:
        return []
    tagged_tokens = pos_tag(tokens)
    return [
        lemmatizer.lemmatize(token, pos=get_wordnet_pos(tag))
        for token, tag in tagged_tokens
    ]


news["tokens_lemmatized"] = news["tokens_no_stopwords"].apply(lemmatize_tokens)
news["text_lemmatized"] = news["tokens_lemmatized"].apply(" ".join)

news[["text_no_stopwords", "text_lemmatized"]].head(3)


,text_no_stopwords,text_lemmatized
0,central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...,central bank bring gold reserve home asgeopoliticalrisks rise central bank bring gold reserve home asgeopoliticalris...
1,oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...,oil rise percent white house u iran talk happen oil rise percent white house u iran talk happen oil price rise thurs...
2,jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...,jamie dimon market underestimate risk buy stock treasurys current jamie dimon market underestimate risk buy stock tr...


# Penyelarasan Data

In [20]:
#change timezone to WIB
news['published_raw'] = pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
news['published_raw'] = news['published_raw'].dt.tz_convert('Asia/Jakarta')

#pisahin tanggal & waktu
news['date'] = news['published_raw'].dt.date
news['time'] = news['published_raw'].dt.time

In [21]:
#change timezone to WIB
news['published_raw'] = pd.to_datetime(news['published_raw'], errors='coerce', utc=True)
news['published_raw'] = news['published_raw'].dt.tz_convert('Asia/Jakarta')

#pisahin tanggal & waktu
news['date'] = news['published_raw'].dt.date
news['time'] = news['published_raw'].dt.time

In [22]:
def align_to_trading_date(dt):
    # Step 1: kalau lewat cutoff 15:15:00, geser ke hari berikutnya
    cutoff = dt.replace(hour=15, minute=0, second=0, microsecond=0)
    if dt > cutoff:
        dt = dt + pd.Timedelta(days=1)

    # Step 2: kalau hasilnya jatuh di weekend, geser ke Senin
    if dt.weekday() == 5:      # Sabtu
        dt = dt + pd.Timedelta(days=2)
    elif dt.weekday() == 6:    # Minggu
        dt = dt + pd.Timedelta(days=1)

    return dt.normalize()  # buang jam, sisakan tanggal saja

news['aligned_trading_date'] = news['date_raw_dt'].apply(align_to_trading_date)
news

,keyword_matched,title,url,preview_summary,full_text,section,published_raw,published_date,published_time,published_timezone,...,normalized_text,lower_text,detected_lang,tokens_no_stopwords,text_no_stopwords,tokens_lemmatized,text_lemmatized,date,time,aligned_trading_date
0,geopolitical risk,Central banks are bringing gold reserves home asgeopoliticalrisks rise,https://www.cnbc.com/2026/06/17/central-banks-gold-reserves-domestic-storage.html?&qsearchterm=geopolitical risk,"More and more central banks are storing gold bullion at home rather than overseas, as they expect to buy more of the...","In this article More and more central banks are storing gold bullion at home rather than overseas, as they expect to...",Gold,2026-06-17 14:28:36+07:00,2026-06-17,07:28 AM,0.0,...,Central banks are bringing gold reserves home asgeopoliticalrisks rise Central banks are bringing gold reserves home...,central banks are bringing gold reserves home asgeopoliticalrisks rise central banks are bringing gold reserves home...,en,"[central, banks, bringing, gold, reserves, home, asgeopoliticalrisks, rise, central, banks, bringing, gold, reserves...",central banks bringing gold reserves home asgeopoliticalrisks rise central banks bringing gold reserves home asgeopo...,"[central, bank, bring, gold, reserve, home, asgeopoliticalrisks, rise, central, bank, bring, gold, reserve, home, as...",central bank bring gold reserve home asgeopoliticalrisks rise central bank bring gold reserve home asgeopoliticalris...,2026-06-17,14:28:36,2026-06-17
1,geopolitical risk,Oil rises 2% as White House says no US-Iran talks happening,https://www.cnbc.com/2026/08/27/oil-prices-extend-losses-on-expectations-talks-to-ease-middle-east-supply-woes.html?...,"Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomatic efforts by ot...","In this article Oil prices rose Thursday, after Washington confirmed it was not in talks with ‌Iran despite diplomat...",Oil,2026-08-27 09:49:03+07:00,2026-08-27,02:49 AM,0.0,...,Oil rises 2 percent as White House says no US Iran talks happening Oil rises 2 percent as White House says no US Ira...,oil rises 2 percent as white house says no us iran talks happening oil rises 2 percent as white house says no us ira...,en,"[oil, rises, percent, white, house, us, iran, talks, happening, oil, rises, percent, white, house, us, iran, talks, ...",oil rises percent white house us iran talks happening oil rises percent white house us iran talks happening oil pric...,"[oil, rise, percent, white, house, u, iran, talk, happen, oil, rise, percent, white, house, u, iran, talk, happen, o...",oil rise percent white house u iran talk happen oil rise percent white house u iran talk happen oil price rise thurs...,2026-08-27,09:49:03,2026-08-27
2,geopolitical risk,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,https://www.cnbc.com/2026/07/21/jpmorgan-chase-ceo-jamie-dimon-market-risk.html?&qsearchterm=geopolitical risk,JPMorgan Chase CEO Jamie Dimon said investors are underestimating the risks facing the global economy and that he wo...,In this article JPMorgan ChaseCEOJamie Dimonsaid investors are underestimating the risks facing the global economy a...,Finance,2026-07-21 06:01:01+07:00,2026-07-20,11:01 PM,0.0,...,Jamie Dimon says markets underestimate risks and he wouldn t buy stocks or Treasurys at current Jamie Dimon says mar...,jamie dimon says markets underestimate risks and he wouldn t buy stocks or treasurys at current jamie dimon says mar...,en,"[jamie, dimon, markets, underestimate, risks, buy, stocks, treasurys, current, jamie, dimon, markets, underestimate,...",jamie dimon markets underestimate risks buy stocks treasurys current jamie dimon markets underestimate risks buy sto...,"[jamie, dimon, market, underestimate, risk, buy, stock, treasurys, current, jamie, dimon, market, underestimate, ris...",jamie dimon market underestimate risk buy stock treasurys curre

In [23]:
daily_news = news.groupby('aligned_trading_date').agg(
    news_count=('title', 'count'),
    all_titles=('title', lambda x: ' | '.join(x)),
    all_text_lemmatized=('text_lemmatized', lambda x: ' | '.join(x))
).reset_index()

In [24]:
merged_data = pd.merge(
    bi_raw,
    daily_news,
    left_on='Tanggal_dt',                   
    right_on='aligned_trading_date',   
    how='left'
)

merged_data

,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal,aligned_trading_date,news_count,all_titles,all_text_lemmatized
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01,2026-09-01,1.0,Oil prices rise as latest fighting resurrects Middle East supply disruption risks,oil price rise late fight resurrects middle east supply disruption risk oil price rise late fight resurrects middle ...
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31,2026-08-31,7.0,"U.S. Treasury’s Bessent faces G20 diplomacy test amid tariffs, Iran war, bond turmoil | Russia preparing ‘massive st...",treasury bessent face g20 diplomacy test amid tariff iran war bond turmoil treasury bessent face g20 diplomacy test ...
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28,2026-08-28,6.0,China’s super-rich fled Singapore. Now they want to come back | Trump’s Greenland fixation puts security at the hear...,china super rich flee singapore want come back china super rich flee singapore want come back year ago wealthy chine...
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM,2026-08-27,2026-08-27,1.0,Oil rises 2% as White House says no US-Iran talks happening,oil rise percent white house u iran talk happen oil rise percent white house u iran talk happen oil price rise thurs...
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM,2026-08-26,2026-08-26,4.0,The ‘debasement trade’ returns after Bessent bond maneuver. Crypto and gold are back in style | World Liberty Financ...,debasement trade return bessent bond maneuver crypto gold back style debasement trade return bessent bond maneuver c...
...,...,...,...,...,...,...,...,...,...,...,...,...
1197,1198,1,14310.20,14167.81,2021-09-07,09/07/2021,12:00 AM,2021-09-07,NaT,NaN,NaN,NaN
1198,1199,1,14332.31,14189.70,2021-09-06,09/06/2021,12:00 AM,2021-09-06,2021-09-06,1.0,"Inflation could repeat the 1960s, when the Fed lost control, Niall Ferguson says",inflation repeat 1960s fed lose control niall ferguson inflation repeat 1960s fed lose control niall ferguson inflat...
1199,1200,1,14352.41,14209.60,2021-09-03,09/03/2021,12:00 AM,2021-09-03,NaT,NaN,NaN,NaN
1200,1201,1,14355.42,14212.58,2021-09-02,09/02/2021,12:00 AM,2021-09-02,NaT,NaN,NaN,NaN


In [25]:
# --- CEK HASIL CUTOFF (pakai date_raw_dt yang sudah WIB) ---
news['jam'] = news['date_raw_dt'].dt.time
news['weekday'] = news['date_raw_dt'].dt.day_name()
news['lewat_cutoff'] = news['date_raw_dt'].dt.time > pd.Timestamp("15:15:00").time()

# Tampilkan berita yang persis di sekitar jam cutoff (14:50 - 15:30) untuk dicek manual
mask_sekitar_cutoff = news['jam'].apply(lambda t: (t.hour == 15 and t.minute <= 30) or (t.hour == 14 and t.minute >= 50))
cek_cutoff = news.loc[mask_sekitar_cutoff, ['title', 'date_raw_dt', 'jam', 'weekday', 'lewat_cutoff', 'aligned_trading_date']]
print(cek_cutoff.sort_values('date_raw_dt').to_string(index=False))

# Cek kasus weekend shifting (berita Jumat setelah cutoff -> harus jadi Senin)
mask_weekend_case = news['weekday'].isin(['Friday', 'Saturday', 'Sunday'])
cek_weekend = news.loc[mask_weekend_case, ['title', 'date_raw_dt', 'weekday', 'lewat_cutoff', 'aligned_trading_date']]
print(cek_weekend.sort_values('date_raw_dt').to_string(index=False))

                                                                                                         title         date_raw_dt      jam   weekday  lewat_cutoff aligned_trading_date
                                        Facebook whistleblower behind major leak is going to testify in Europe 2021-10-12 14:58:07 14:58:07   Tuesday         False           2021-10-12
                                                    What is Putin’s greatest worry right now? His own citizens 2021-10-14 15:17:54 15:17:54  Thursday          True           2021-10-15
                                               10-year Treasury yield dips as bond market volatility continues 2021-10-29 15:08:09 15:08:09    Friday         False           2021-11-01
               ‘Zero trust’ and mutual dislike: Why hopes to resolve U.S.-Russia tensions are low as talks ... 2022-01-10 15:13:44 15:13:44    Monday         False           2022-01-11
                                                      Oil edges higher on t

In [26]:
tanggal_dicek = cek_cutoff['aligned_trading_date'].unique()

for tgl in tanggal_dicek:
    subset = merged_data[merged_data['aligned_trading_date'] == tgl]
    print(f"\n=== Trading date: {tgl.date()} ===")
    print(f"Jumlah berita ter-merge: {subset['news_count'].values}")
    # tampilkan judul2 berita yg masuk ke tanggal ini dari df news asli
    judul_masuk = news[news['aligned_trading_date'] == tgl][['title', 'published_raw']]
    print(judul_masuk.sort_values('published_raw').to_string(index=False))


=== Trading date: 2026-07-07 ===
Jumlah berita ter-merge: [3.]
                                                                             title             published_raw
Trump’s calls, Ukraine’s strikes and Russia’s barrage on Kyiv put markets on alert 2026-07-06 15:27:36+07:00
                     Oil prices gain as focus shifts to supply recovery and demand 2026-07-07 08:10:02+07:00
                            Dollar higher after data as yen holds near 40-year low 2026-07-07 08:16:23+07:00

=== Trading date: 2026-06-08 ===
Jumlah berita ter-merge: [10.]
                                                                                 title             published_raw
10-year Treasury yield surges above 4.53% as hot jobs report dents hopes for rate cuts 2026-06-05 15:05:07+07:00
                    Bitcoin cracks $60,000, sinking to lowest level since October 2024 2026-06-05 18:41:45+07:00
    Hot jobs report puts Fed cuts further out of reach as Chair Warsh faces policy ... 2026-06-0

# Ekstraksi Fitur NLP


In [27]:
from __future__ import annotations

import re
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pysentiment2 as ps
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# 1. Sentimen berbasis Lexicon

In [28]:
# 1. VADER (general-purpose)
_vader = SentimentIntensityAnalyzer()
def compute_vader_sentiment(df: pd.DataFrame, text_col: str = "text_lemmatized") -> pd.DataFrame:

    scores = df[text_col].fillna("").apply(_vader.polarity_scores).apply(pd.Series)
    scores = scores.rename(columns={
        "neg": "vader_neg", "neu": "vader_neu",
        "pos": "vader_pos", "compound": "vader_compound",
    })
    return pd.concat([df.reset_index(drop=True), scores.reset_index(drop=True)], axis=1)


# 2. Loughran-McDonald (finance-specific)
_lm = ps.LM()  # dictionary bundled offline (LM.csv), tidak perlu koneksi internet
def compute_lm_sentiment(df: pd.DataFrame, text_col: str = "text_lemmatized") -> pd.DataFrame:
    def _score(text: str) -> pd.Series:
        tokens = _lm.tokenize(text if isinstance(text, str) else "")
        s = _lm.get_score(tokens)
        return pd.Series({
            "lm_positive": s["Positive"],
            "lm_negative": s["Negative"],
            "lm_polarity": s["Polarity"],
            "lm_subjectivity": s["Subjectivity"],
        })

    scores = df[text_col].apply(_score)
    return pd.concat([df.reset_index(drop=True), scores.reset_index(drop=True)], axis=1)


# 3. Indeks risiko geopolitik berbasis keyword density 
# Daftar sama seperti filter relevansi
DEFAULT_GPR_KEYWORDS = [
    "geopolitics", "geopolitical risk", "geopolitical tensions",
    "geopolitical fragmentation", "armed conflict", "global conflict",
    "international conflict", "military tensions", "national security",
    "sanctions", "trade war", "tariffs", "embargo", "export controls",
    "protectionism", "supply chain disruption", "diplomacy",
    "foreign policy", "nato", "brics", "opec", "g7",
]

def compute_gpr_density(
    df: pd.DataFrame,
    text_col: str = "lower_text",
    keywords: list[str] | None = None,
) -> pd.DataFrame:

    kw = keywords or DEFAULT_GPR_KEYWORDS
    pattern = re.compile("|".join(re.escape(k) for k in kw))

    def _count(text: str) -> int:
        if not isinstance(text, str) or not text:
            return 0
        return len(pattern.findall(text))

    hits = df[text_col].apply(_count)
    word_count = df[text_col].fillna("").str.split().apply(len).replace(0, 1)

    out = df.copy()
    out["gpr_hits"] = hits
    out["gpr_density"] = hits / word_count
    return out

# 4. Vectorization klasik: TF-IDF + SVD (LSA) untuk reduksi dimensi
@dataclass
class TfidfSvdExtractor:
    max_features: int = 5000
    n_components: int = 75
    ngram_range: tuple[int, int] = (1, 2)
    min_df: int = 5
    random_state: int = 42

    _vectorizer: TfidfVectorizer = field(init=False, default=None, repr=False)
    _svd: TruncatedSVD = field(init=False, default=None, repr=False)
    _is_fitted: bool = field(init=False, default=False, repr=False)

    def fit(self, texts: pd.Series) -> "TfidfSvdExtractor":
        self._vectorizer = TfidfVectorizer(
            max_features=self.max_features,
            ngram_range=self.ngram_range,
            min_df=self.min_df,
            sublinear_tf=True,
        )
        tfidf_matrix = self._vectorizer.fit_transform(texts.fillna(""))

        # n_components dibatasi supaya tidak melebihi jumlah fitur/dokumen train
        n_comp = min(self.n_components, tfidf_matrix.shape[1] - 1, tfidf_matrix.shape[0] - 1)
        self._svd = TruncatedSVD(n_components=max(n_comp, 2), random_state=self.random_state)
        self._svd.fit(tfidf_matrix)
        self._is_fitted = True
        return self

    def transform(self, texts: pd.Series) -> pd.DataFrame:
        if not self._is_fitted:
            raise RuntimeError("Panggil .fit(train_texts) dulu sebelum .transform().")
        tfidf_matrix = self._vectorizer.transform(texts.fillna(""))
        reduced = self._svd.transform(tfidf_matrix)
        cols = [f"tfidf_svd_{i}" for i in range(reduced.shape[1])]
        return pd.DataFrame(reduced, columns=cols, index=texts.index)

    @property
    def explained_variance_ratio_(self) -> np.ndarray:
        return self._svd.explained_variance_ratio_


# 5. Agregasi fitur level-artikel -> level-harian (join key: aligned_trading_date)
def aggregate_daily_features(
    article_df: pd.DataFrame,
    date_col: str = "aligned_trading_date",
) -> pd.DataFrame:
    sentiment_cols = [c for c in article_df.columns if c.startswith(("vader_", "lm_"))]
    svd_cols = [c for c in article_df.columns if c.startswith("tfidf_svd_")]
    mean_cols = sentiment_cols + svd_cols + (["gpr_density"] if "gpr_density" in article_df else [])

    agg_dict = {c: "mean" for c in mean_cols}
    if "gpr_hits" in article_df.columns:
        agg_dict["gpr_hits"] = "sum"

    daily = article_df.groupby(date_col).agg(agg_dict)
    daily["news_count"] = article_df.groupby(date_col).size()
    return daily.reset_index()

# 1.1 Fitur Sentimen(Vader + Loughran-McDonald)

In [29]:
news_feat = compute_vader_sentiment(news, text_col='text_lemmatized')
news_feat = compute_lm_sentiment(news_feat, text_col='text_lemmatized')

news_base = compute_vader_sentiment(news, text_col='lower_text')
news_base = compute_lm_sentiment(news_base, text_col='lower_text')

In [30]:
print(f"Jumlah baris feat: {len(news_feat)}")
news_feat[['title', 'vader_compound', 'lm_polarity', 'lm_subjectivity']].head(5)

# Hasil sentimen VADER
vader_feat_result = news_feat[
    [
        "title",
        "vader_neg",
        "vader_neu",
        "vader_pos",
        "vader_compound",
    ]
].copy()



Jumlah baris feat: 4336


In [31]:
vader_feat_result.head(5)

,title,vader_neg,vader_neu,vader_pos,vader_compound
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.055,0.803,0.142,0.9796
1,Oil rises 2% as White House says no US-Iran talks happening,0.145,0.740,0.116,-0.8074
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,0.183,0.693,0.125,-0.9399
3,China’s super-rich fled Singapore. Now they want to come back,0.058,0.692,0.250,0.9991
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,0.151,0.730,0.119,-0.8492


In [32]:
print(f"Jumlah baris lower: {len(news_base)}")
news_base[['title', 'vader_compound', 'lm_polarity', 'lm_subjectivity']].head(5)

# Hasil sentimen VADER
vader_base_result = news_base[
    [
        "title",
        "vader_neg",
        "vader_neu",
        "vader_pos",
        "vader_compound",
    ]
].copy()

vader_base_result.head(5)

Jumlah baris lower: 4336


,title,vader_neg,vader_neu,vader_pos,vader_compound
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.029,0.901,0.071,0.9631
1,Oil rises 2% as White House says no US-Iran talks happening,0.110,0.812,0.078,-0.9474
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,0.097,0.819,0.084,0.3641
3,China’s super-rich fled Singapore. Now they want to come back,0.042,0.795,0.163,0.9991
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,0.112,0.800,0.088,-0.8911


In [33]:
vader_base_result.head(5)

,title,vader_neg,vader_neu,vader_pos,vader_compound
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.029,0.901,0.071,0.9631
1,Oil rises 2% as White House says no US-Iran talks happening,0.110,0.812,0.078,-0.9474
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,0.097,0.819,0.084,0.3641
3,China’s super-rich fled Singapore. Now they want to come back,0.042,0.795,0.163,0.9991
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,0.112,0.800,0.088,-0.8911


In [34]:
comparison = pd.DataFrame({
    "title": news["title"],
    "vader_lower": news_base["vader_compound"],
    "vader_lemma": news_feat["vader_compound"],
    "lm_lower": news_base["lm_polarity"],
    "lm_lemma": news_feat["lm_polarity"],
})

comparison["vader_diff"] = comparison["vader_lower"] - comparison["vader_lemma"]
comparison["lm_diff"] = comparison["lm_lower"] - comparison["lm_lemma"]

comparison.head(5)

,title,vader_lower,vader_lemma,lm_lower,lm_lemma,vader_diff,lm_diff
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.9631,0.9796,-0.555555,-0.555555,-0.0165,0.000000
1,Oil rises 2% as White House says no US-Iran talks happening,-0.9474,-0.8074,-0.111111,-0.111111,-0.1400,0.000000
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,0.3641,-0.9399,-0.461538,-0.538462,1.3040,0.076923
3,China’s super-rich fled Singapore. Now they want to come back,0.9991,0.9991,-0.150000,-0.150000,0.0000,0.000000
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,-0.8911,-0.8492,-0.714286,-0.714286,-0.0419,0.000000


In [35]:
print("Perbedaan VADER:", (comparison["vader_diff"] != 0).sum())
print("Perbedaan LM   :", (comparison["lm_diff"] != 0).sum())

Perbedaan VADER: 4280
Perbedaan LM   : 2374


### Keputusan final sumber teks per lexicon

- **VADER -> `lower_text`**: VADER butuh negasi ("not"), intensifier ("very"), kapitalisasi, dan tanda baca yang hilang saat lemmatization/stopword removal.
- **LM -> `text_lemmatized`**: LM sudah menstem kata secara internal (bag-of-words), jadi cocok dipakai di teks yang sudah dinormalisasi, sejalan dengan `text_lemmatized` yang juga dipakai TF-IDF/SVD nantinya.

In [36]:
# Kombinasi final: VADER dari lower_text, LM dari text_lemmatized
news_feat = compute_vader_sentiment(news, text_col='lower_text')
news_feat = compute_lm_sentiment(news_feat, text_col='text_lemmatized')

print(f"Jumlah baris: {len(news_feat)}")
news_feat[['title', 'vader_compound', 'lm_polarity', 'lm_subjectivity']].head(5)

Jumlah baris: 4336


,title,vader_compound,lm_polarity,lm_subjectivity
0,Central banks are bringing gold reserves home asgeopoliticalrisks rise,0.9631,-0.555555,0.039474
1,Oil rises 2% as White House says no US-Iran talks happening,-0.9474,-0.111111,0.101887
2,Jamie Dimon says markets underestimate risks and he wouldn’t buy stocks or Treasurys at current ...,0.3641,-0.538462,0.104418
3,China’s super-rich fled Singapore. Now they want to come back,0.9991,-0.150000,0.096154
4,‘Don’t get too comfortable’: Wall Street’s ‘fear gauge’ hits 2026 low — here’s why it’s ...,-0.8911,-0.714286,0.083665


# 1.2 Indeks Resiko Geopolitik berbasis Keyword

In [37]:
news_feat = compute_gpr_density(news_feat, text_col='lower_text', keywords=DEFAULT_GPR_KEYWORDS)

print(news_feat['gpr_hits'].describe())
news_feat[['title', 'gpr_hits', 'gpr_density']].sort_values('gpr_density', ascending=False).head(5)

count    4336.000000
mean        2.711946
std         3.997439
min         0.000000
25%         0.000000
50%         1.000000
75%         3.000000
max        41.000000
Name: gpr_hits, dtype: float64


,title,gpr_hits,gpr_density
3818,"Emerging economies group BRICS invites 6 new members, including Saudi Arabia and Iran",32,0.035049
3652,"Oil steadies in choppy trading on tariff uncertainty, OPEC+ hike plans",17,0.031540
2756,OPEC+ to raise oil output slightly even as U.S.-Israel strikes on Iran disrupt shipments,14,0.030172
2285,An emboldened NATO provides a bittersweet moment for war-torn Ukraine,29,0.029502
1724,OPEC+ keeps oil output steady amid turmoil among members,13,0.029279


# 1.3 TF-IDF + SVD

In [38]:
extractor = TfidfSvdExtractor(max_features=5000, n_components=75, min_df=5)
extractor.fit(news_feat['text_lemmatized'])

svd_vec = extractor.transform(news_feat['text_lemmatized'])
news_feat = pd.concat([news_feat.reset_index(drop=True), svd_vec.reset_index(drop=True)], axis=1)

print(f"Explained variance ratio (kumulatif, {extractor.n_components} komponen): {extractor.explained_variance_ratio_.sum():.3f}")
svd_vec.head(3)

Explained variance ratio (kumulatif, 75 komponen): 0.237


,tfidf_svd_0,tfidf_svd_1,tfidf_svd_2,tfidf_svd_3,tfidf_svd_4,tfidf_svd_5,tfidf_svd_6,tfidf_svd_7,tfidf_svd_8,tfidf_svd_9,...,tfidf_svd_65,tfidf_svd_66,tfidf_svd_67,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74
0,0.272386,0.130731,-0.060716,0.081283,-0.011045,-0.031668,-0.045758,-0.011606,-0.022026,-0.049104,...,-0.079944,-0.105850,0.020177,-0.112659,0.005519,0.085705,-0.042776,0.003524,-0.045443,-0.030497
1,0.340730,0.007782,0.270065,-0.044959,0.044673,0.062974,0.041566,0.118258,-0.002489,-0.011118,...,-0.029576,0.001795,-0.007135,-0.039125,-0.025280,-0.024280,0.023515,0.040962,-0.008034,-0.002545
2,0.296977,0.131154,-0.088787,0.032396,-0.042011,-0.037674,-0.037804,0.063501,0.000654,-0.034595,...,0.054777,-0.074153,-0.012344,0.020130,0.053371,0.052721,-0.003671,-0.008719,0.029853,0.039147


# 1.4 Agregaris ke Level Harian & Merge ke Data Kurs

daily_features = aggregate_daily_features(news_feat, date_col='aligned_trading_date')
print(f"Jumlah hari trading dengan fitur NLP: {len(daily_features)}")
daily_features.head(5)

In [39]:
daily_features = aggregate_daily_features(
    news_feat,
    date_col="aligned_trading_date",
)

print(f"Jumlah hari trading dengan fitur NLP: {len(daily_features)}")
daily_features.head()

Jumlah hari trading dengan fitur NLP: 1156


,aligned_trading_date,vader_neg,vader_neu,vader_pos,vader_compound,lm_positive,lm_negative,lm_polarity,lm_subjectivity,tfidf_svd_0,...,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74,gpr_density,gpr_hits,news_count
0,2021-09-06,0.0700,0.881,0.04900,-0.9505,6.00,21.0,-0.555556,0.106299,0.303250,...,0.005938,-0.019196,-0.025434,0.074565,-0.001749,0.056623,-0.004710,0.000000,0,1
1,2021-09-08,0.0830,0.837,0.07900,-0.3395,11.00,43.0,-0.592593,0.142105,0.285621,...,0.024561,-0.027375,0.006369,0.008435,-0.034538,-0.048796,0.015683,0.001230,1,1
2,2021-09-09,0.0740,0.799,0.12700,0.9953,9.00,25.0,-0.470588,0.083744,0.312970,...,-0.004390,-0.003963,-0.013189,0.038953,0.011739,0.049731,-0.012375,0.006303,6,1
3,2021-09-10,0.0470,0.888,0.06500,0.8918,3.00,11.0,-0.571429,0.067308,0.278243,...,0.028025,0.036017,-0.008151,-0.009683,0.000592,0.004094,-0.050824,0.006612,4,1
4,2021-09-13,0.0965,0.825,0.07875,-0.0011,8.75,37.0,-0.595140,0.092247,0.304775,...,0.001062,-0.019711,0.001495,-0.049784,0.023224,0.047122,0.028836,0.001365,6,4


In [40]:
merged_data_v2 = pd.merge(
    bi_raw,
    daily_features,
    left_on='Tanggal_dt',
    right_on='aligned_trading_date',
    how='left',
)

print(f"Baris merged_data_v2: {len(merged_data_v2)}")
print(f"Hari tanpa berita (NaN fitur NLP): {merged_data_v2['news_count'].isna().sum()}")
merged_data_v2.head(5)

Baris merged_data_v2: 1202
Hari tanpa berita (NaN fitur NLP): 142


,NO,Nilai,Kurs Jual,Kurs Beli,Tanggal_dt,tanggal,jam,Tanggal,aligned_trading_date,vader_neg,...,tfidf_svd_68,tfidf_svd_69,tfidf_svd_70,tfidf_svd_71,tfidf_svd_72,tfidf_svd_73,tfidf_svd_74,gpr_density,gpr_hits,news_count
0,1,1,17834.73,17657.27,2026-09-01,09/01/2026,12:00 AM,2026-09-01,2026-09-01,0.157000,...,-0.015221,-0.024285,0.035738,-0.000381,0.030808,-0.005420,-0.010191,0.002008,1.0,1.0
1,2,1,17791.51,17614.49,2026-08-31,08/31/2026,12:00 AM,2026-08-31,2026-08-31,0.112000,...,0.001745,0.021222,-0.005325,-0.024874,0.018635,0.014870,0.013658,0.006466,43.0,7.0
2,3,1,17850.81,17673.19,2026-08-28,08/28/2026,12:00 AM,2026-08-28,2026-08-28,0.084333,...,0.019693,-0.006501,-0.020886,-0.015738,0.009945,0.008294,0.014060,0.008579,37.0,6.0
3,4,1,17805.58,17628.42,2026-08-27,08/27/2026,12:00 AM,2026-08-27,2026-08-27,0.110000,...,-0.039125,-0.025280,-0.024280,0.023515,0.040962,-0.008034,-0.002545,0.008432,5.0,1.0
4,5,1,17791.51,17614.49,2026-08-26,08/26/2026,12:00 AM,2026-08-26,2026-08-26,0.073000,...,0.007015,0.020591,-0.002135,0.024654,0.011187,0.004372,0.043154,0.004193,10.0,4.0


# 1.5 Sanity Check

In [41]:
check = merged_data_v2.sort_values('Tanggal_dt').copy()
check['kurs_next'] = check['Kurs Jual'].shift(-1)
check['kurs_change_next'] = check['kurs_next'] - check['Kurs Jual']

corr_cols = ['vader_compound', 'lm_polarity', 'gpr_density', 'gpr_hits', 'news_count', 'kurs_change_next']
check[corr_cols].corr()['kurs_change_next']

vader_compound     -0.047843
lm_polarity        -0.018804
gpr_density         0.045604
gpr_hits            0.054945
news_count          0.036842
kurs_change_next    1.000000
Name: kurs_change_next, dtype: float64